# lengua-ligera: el molde en TPU (Colab)
Entorno de ejecución → Cambiar tipo → **TPU v5e-1**.

Antes: sube `lengua-ligera-base.zip` a tu Google Drive en la carpeta `lengua-ligera/`. Los `.llg` y logs se guardan ahí mismo (sobreviven a desconexiones).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/ll/data && cd /content/ll && unzip -o -q /content/drive/MyDrive/lengua-ligera/lengua-ligera-base.zip && mv -f vocab.txt valid.bin data/
import jax; print(jax.__version__, jax.devices())

## 1. Corpus (~1.5 mil millones de caracteres). Se rehace en cada sesión nueva (15-30 min).

In [ ]:
!cd /content/ll && DATA=/content/ll/data TRAIN_CHARS=1500000000 python molde.py corpus

## 2. Prueba de velocidad (200 pasos). Copia la salida a Claude.

In [ ]:
!cd /content/ll && DATA=/content/ll/data D=512 L=8 H=8 CTX=256 BATCH=128 STEPS=200 WARMUP=50 EVAL_EVERY=200 EVAL_CHARS=200000 OUT=/content/ll/prueba.llg python molde.py train

## 3. Molde float (d512 L8, ~25M parámetros). Guarda el mejor .llg en Drive cada 2000 pasos.

In [ ]:
!cd /content/ll && DATA=/content/ll/data D=512 L=8 H=8 CTX=256 BATCH=128 STEPS=40000 LR=1e-3 WARMUP=500 EVAL_EVERY=2000 OUT=/content/drive/MyDrive/lengua-ligera/float_d512.llg python molde.py train 2>&1 | tee /content/drive/MyDrive/lengua-ligera/float_d512.log

## 4. QAT ternario desde el float

In [ ]:
!cd /content/ll && DATA=/content/ll/data INIT=/content/drive/MyDrive/lengua-ligera/float_d512.llg QAT=ternario-g32 STEPS=12000 LR=1e-3 WARMUP=100 EVAL_EVERY=1000 OUT=/content/drive/MyDrive/lengua-ligera/tern_d512.llg python molde.py train 2>&1 | tee /content/drive/MyDrive/lengua-ligera/tern_d512.log

## 5. Control justo: float con los mismos pasos extra

In [ ]:
!cd /content/ll && DATA=/content/ll/data INIT=/content/drive/MyDrive/lengua-ligera/float_d512.llg STEPS=12000 LR=1e-3 WARMUP=100 EVAL_EVERY=1000 OUT=/content/drive/MyDrive/lengua-ligera/ctrl_d512.llg python molde.py train 2>&1 | tee /content/drive/MyDrive/lengua-ligera/ctrl_d512.log